In [7]:
# !pip install cdsapi
# !pip install eccodes
# !pip install cfgrib eccodes
# !conda install -c conda-forge eccodes cfgrib xarray -y

In [2]:
import cdsapi

dataset = "reanalysis-era5-pressure-levels"
request = {
    "product_type": ["reanalysis"],
    "variable": ["temperature"],
    "year": ["2004"],
    "month": [
        "01", "02", "03",
        "04", "05", "06",
        "07", "08", "09",
        "10", "11", "12"
    ],
    "day": [
        "01", "02", "03",
        "04", "05", "06",
        "07", "08", "09",
        "10", "11", "12",
        "13", "14", "15",
        "16", "17", "18",
        "19", "20", "21",
        "22", "23", "24",
        "25", "26", "27",
        "28", "29", "30",
        "31"
    ],
    "time": [
        "00:00", "06:00", "12:00",
        "18:00"
    ],
    "pressure_level": ["925", "1000"],
    "data_format": "grib",
    "download_format": "unarchived",
    "area": [52.54000, 16.80000, 52.28000, 17.06000]
}

client = cdsapi.Client(
    url = "https://cds.climate.copernicus.eu/api",
    key = "f55a5040-c43c-40a7-957a-f078b44210f3"
)
client.retrieve(dataset, request).download()


2026-04-10 20:02:29,001 INFO Request ID is 03a0b366-d9a8-4486-a59f-9c3d849ac86c
2026-04-10 20:02:29,324 INFO status has been updated to accepted


KeyboardInterrupt: 

In [13]:
import eccodes
import xarray as xr
import cfgrib

# Odczyt pliku GRIB przez xarray
ds = xr.open_dataset(
    '55d23fc61d0eb2bbfbf4c44baffbc619.grib',
    engine='cfgrib'
)

print(ds)

<xarray.Dataset> Size: 35kB
Dimensions:        (time: 1464, isobaricInhPa: 2, latitude: 1, longitude: 1)
Coordinates:
  * time           (time) datetime64[ns] 12kB 2004-01-01 ... 2004-12-31T18:00:00
    valid_time     (time) datetime64[ns] 12kB ...
  * isobaricInhPa  (isobaricInhPa) float64 16B 1e+03 925.0
  * latitude       (latitude) float64 8B 52.5
  * longitude      (longitude) float64 8B 17.0
    number         int64 8B ...
    step           timedelta64[ns] 8B ...
Data variables:
    t              (time, isobaricInhPa, latitude, longitude) float32 12kB ...
Attributes:
    GRIB_edition:            1
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-04-04T17:04 GRIB to CDM+CF via cfgrib-0.9.1...


In [15]:
import pandas as pd

df = ds.to_dataframe().reset_index()


# Wyświetl wszystkie wiersze i kolumny
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print(df.shape)  # ile wierszy x kolumn
df.head()

(2928, 8)


,time,isobaricInhPa,latitude,longitude,number,step,valid_time,t
0,2004-01-01 00:00:00,1000.0,52.5,17.0,0,0 days,2004-01-01 00:00:00,271.575439
1,2004-01-01 00:00:00,925.0,52.5,17.0,0,0 days,2004-01-01 00:00:00,268.689941
2,2004-01-01 06:00:00,1000.0,52.5,17.0,0,0 days,2004-01-01 06:00:00,270.003174
3,2004-01-01 06:00:00,925.0,52.5,17.0,0,0 days,2004-01-01 06:00:00,267.951904
4,2004-01-01 12:00:00,1000.0,52.5,17.0,0,0 days,2004-01-01 12:00:00,272.496582


In [17]:
# Rozdzielam na dwa DataFrame 
df_925 = df[df['isobaricInhPa'] == 925][['time','t']].rename(columns = {'t': 't_925'})
df_1000 = df[df['isobaricInhPa'] == 1000][['time','t']].rename(columns = {'t': 't_1000'})

# Łaczymy oba df, w jeden do obliczenia różnicy
df_delta = pd.merge(df_925, df_1000, how='inner', on='time')

# kolumna t_delta = podaje rożnice w stopniach 
df_delta['t_delta'] = df_delta['t_1000']-df_delta['t_925']
df_delta['inversion_0/1'] = [1 if e > 0 else 0 for e in df_delta['t_delta']]

# wyciagamy sama date , do łatiwejszegeo grupowania i wyciagniecia dziennej sreniej 

df_delta['data'] = df_delta['time'].dt.date
df_delta.tail()

,time,t_925,t_1000,t_delta,inversion_0/1,data
1459,2004-12-30 18:00:00,273.607178,275.510498,1.903320,1,2004-12-30
1460,2004-12-31 00:00:00,276.000977,275.960205,-0.040771,0,2004-12-31
1461,2004-12-31 06:00:00,276.012451,276.914307,0.901855,1,2004-12-31
1462,2004-12-31 12:00:00,275.958252,277.095459,1.137207,1,2004-12-31
1463,2004-12-31 18:00:00,276.162354,277.588379,1.426025,1,2004-12-31


In [19]:
df_delta.to_csv('/Users/marcinurbanski/Desktop/merito_projekt_pliki_robocze/qualityair_projekt/ready_raw_data/inwersja_pz_2004.csv')

In [9]:
# import pandas as pd

# # Konwersja do DataFrame (temperatura w Kelvinach -> Celsjusze)
# df = ds['t'].to_dataframe().reset_index()
# df['temperature_C'] = df['t'] - 273.15

# print(df.head(20))
# print(f"\nZakres dat: {df['time'].min()} → {df['time'].max()}")
# print(f"Poziomy ciśnienia: {df['isobaricInhPa'].unique()}")

                  time  isobaricInhPa  number   step  latitude  longitude  \
0  2024-01-01 00:00:00         1000.0       0 0 days      53.5       14.5   
1  2024-01-01 00:00:00          925.0       0 0 days      53.5       14.5   
2  2024-01-01 06:00:00         1000.0       0 0 days      53.5       14.5   
3  2024-01-01 06:00:00          925.0       0 0 days      53.5       14.5   
4  2024-01-01 12:00:00         1000.0       0 0 days      53.5       14.5   
5  2024-01-01 12:00:00          925.0       0 0 days      53.5       14.5   
6  2024-01-01 18:00:00         1000.0       0 0 days      53.5       14.5   
7  2024-01-01 18:00:00          925.0       0 0 days      53.5       14.5   
8  2024-01-02 00:00:00         1000.0       0 0 days      53.5       14.5   
9  2024-01-02 00:00:00          925.0       0 0 days      53.5       14.5   
10 2024-01-02 06:00:00         1000.0       0 0 days      53.5       14.5   
11 2024-01-02 06:00:00          925.0       0 0 days      53.5       14.5   